## Gemini Master Instructions for Modular Colab **Workflows**

You are helping design and maintain **modular Google Colab workflows** orchestrated by a **central master pipeline**. Every notebook step must remain self-contained, reusable, and easy to debug independently, while still fitting into a larger end-to-end workflow. Make sure to read and follow the guidance below.

---

### 1. Core Architecture Principles

* Build workflows as **modular processing blocks**.
* Each major step must be **self-contained** with clear inputs/outputs.
* Use a **single master pipeline block** to orchestrate execution.
* Avoid tight coupling between modules.
* Prefer explicit data handoffs (dataframes, return values, or defined globals).
* Design modules so they can be **tested independently**.

---

### 2. Variable Management Rules

* **Consolidate all variables within the master pipeline block.**
* If variables are needed elsewhere, **import them as globals**.
* Do not scatter configuration values across cells.
* Avoid duplicate constants inside modules.
* Add new variables to the master pipeline first, then wire downstream.

---

### 3. Notes and Commentary

* **Preserve all notes exactly where they are placed.**
* Do not remove or rewrite notes unless explicitly instructed.
* Treat markdown and comments as long-term documentation.
* Flag outdated notes instead of deleting them.

---

### 4. Notebook Structure

Preferred order:

1. Preflight / setup
2. Authentication / mounts
3. Shared imports
4. Module sections
5. Validation / diagnostics
6. Export / outputs
7. Master pipeline
8. Utility / recovery helpers

Use clear section headers (e.g., `### Preflight Check`, `# Master Pipeline`).

---

### 5. Module Design Requirements

* Each module should have **one responsibility**.
* Wrap logic in clearly named functions.
* Define inputs and outputs explicitly.
* Avoid hidden dependencies.
* Include lightweight validation.
* Document side effects (exports, file moves, etc.).

---

## 6. Master Pipeline Responsibilities

The master pipeline must:

* Define all variables and configuration
* Control execution order
* Pass configuration to modules
* Handle global state intentionally
* Manage logging and status
* Coordinate exports and failure handling

---

### 7. Globals Usage Rules

* Use globals only for intentionally shared configuration.
* Assign globals in the master pipeline.
* Avoid implicit globals in modules.
* Make dependencies on globals explicit.

---

### 8. Imports and Dependencies

* Keep imports organized.
* Avoid unnecessary duplication.
* Include imports in modules only if needed for isolation.
* Do not introduce unnecessary libraries.

---

### 9. Validation and Debugging

* Add validation checkpoints after transformations.
* Include diagnostics (row counts, schema checks, etc.).
* Print clear status messages.
* Fail gracefully where possible.

---

### 10. Output and Export Standards

* Keep export logic in a dedicated section.
* Use clear, traceable naming conventions.
* Avoid hidden output paths.
* Document outputs clearly.

---

### 11. Recovery and Utility Cells

* Keep utilities separate from core workflow.
* Clearly label recovery logic.
* Preserve existing utility cells.

---

### 12. Change Management

* Preserve structure unless improvement is necessary.
* Do not collapse modular design.
* Prefer targeted edits.
* Explain structural changes when needed.

---

### 13. Coding Style

* Write readable, maintainable code.
* Use clear function names.
* Prefer explicit logic over shortcuts.
* Preserve dataframe clarity.

---

### 14. Interaction Rules

When building workflows:

* Assume modular architecture is required.
* Place variables in the master pipeline.
* Preserve notes and structure.
* Return code ready for direct cell insertion.
* Highlight impacted sections when making changes.

---

### Short Instruction Block (Reusable)

```text
Build this Colab workflow using a modular notebook architecture.

Rules:
1. Consolidate all variables within the master pipeline block.
2. Import shared variables as globals when needed.
3. Preserve all notes and markdown exactly as placed.
4. Keep each module self-contained and reusable.
5. Use a master pipeline for orchestration.
6. Do not scatter configuration values.
7. Maintain clear section headers.
8. Separate validation, export, and recovery logic.
9. Prefer targeted updates over rewrites.
10. Write maintainable, debuggable code.
```


### DATA EXTRACTION LOGIC

Do not use hard-coded row numbers or fixed positional logic when parsing these reports. Instead, use anchor-based detection by defining constants for known header or label text, such as const headers = ['Occ (%)', 'Index (MPI)', ...], and locate rows dynamically based on those anchors. This ensures the parser remains stable even if rows shift between properties, report versions, or months.

The parsing logic should always identify the relevant section by searching for the expected text labels in the sheet, rather than assuming a metric will always appear on the same row. Build the mapping from those discovered anchor points, then derive the related values relative to the matched labels. This makes the pipeline more resilient and reduces breakage when report formatting changes.

Use anchor-based parsing only. Never rely on fixed row indexes for report extraction. Define reusable constants for expected labels and headers, scan the sheet to find those anchors, and build mappings from the discovered positions. This ensures the parser continues to work even when report layouts shift.

# Success Critera Test

To deterime if the merge was successful without affecting the underlaying data integrity this test made and must pass before considering success.

Display a Dataframe with the following totals, filtered by Date + Reservations Status for the exported file.

---

## Filters
Filters
Date = `01-01-2026` - `01-31-2026`
Reservation Status = `CHECKEDOUT`

---

## Sum Values
Sum Col `Sold`
Sum Col `Room Revenue`

---

## Criteria
Total for `Sum` must equal **2235**
Total for `Room` Revenue must equal **175872**

# Setup & Auth

In [ ]:
# @title Connect to Google Drive {"vertical-output":true,"single-column":true,"display-mode":"code"}

from google.colab import drive
import os

def setup_environment(source_path, next_path):
    """
    Module: Setup Environment
    Mounts Google Drive, defines global directory variables, and ensures all
    required subfolders exist.
    """
    print("--- Initializing Environment ---")

    # 1. Mount Drive (with a safety catch)
    try:
        drive.mount('/content/drive', force_remount=True)
        print("v Drive mounted successfully.")
    except Exception as e:
        print(f"x Manual action required: Please click the Drive icon to mount. Error: {e}")

    # 2. Define global variables so the rest of the notebook can use them
    global SOURCE_DIR, NEW_DIR, PROCESSED_DIR, EXPORT_DIR, FAILED_DIR, NEXT_DIR, PROJECT_ID, BQ_TABLE_ID

    # 3. Assign Paths
    SOURCE_DIR = source_path

    NEW_DIR = os.path.join(SOURCE_DIR, "data_upload")
    PROCESSED_DIR = os.path.join(SOURCE_DIR, "data_processed")
    EXPORT_DIR = os.path.join(SOURCE_DIR, "data_export")
    FAILED_DIR = os.path.join(SOURCE_DIR, "data_failed")
    NEXT_DIR = os.path.join(next_path, "data_upload")

    # BigQuery Configuration
    PROJECT_ID = "dovetailco"
    BQ_TABLE_ID = "dovetailco.stg.pms_reservations"

    # 4. Create directories dynamically if they don't exist
    directories = [NEW_DIR, PROCESSED_DIR, EXPORT_DIR, FAILED_DIR, NEXT_DIR]

    for directory in directories:
        if not os.path.exists(directory):
            os.makedirs(directory)
            print(f"  -> Created directory: {directory}")

    print(f"v Setup complete. Checking for new files in: {NEW_DIR}\n")

In [ ]:
import requests
import types

# EXTERNAL HELPERS (Standardizer) ---
# Fetch the external column mapping library from GitHub
url = "https://raw.githubusercontent.com/REVREBEL/Metrics-Dataform/main/notebooks/utilities/revrebel_column_standardizer.py"
response = requests.get(url)

if response.status_code == 200:
    module_code = response.text
    revrebel_standardizer = types.ModuleType("revrebel_column_standardizer")
    exec(module_code, revrebel_standardizer.__dict__)
    print("Successfully loaded revrebel_column_standardizer library.")
else:
    print(f"Failed to load external helper. Status code: {response.status_code}")

Successfully loaded revrebel_column_standardizer library.


# Process Headers

In [ ]:
def find_header_row(file_path):
    """Reads the beginning of a file to find the row index where the header likely starts."""
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            if any(keyword in line.lower() for keyword in ['stay_date', 'confirmation', 'rate_code', 'room']):
                return i
    return 0

def drop_unnecessary_columns(df):
    """Removes specific internal and metadata columns. Targeted at snake_case library output."""
    cols_to_drop = [
        'hotel_code', 'market_id', 'source_id', 'booking_origin_id',
        'segment_id', 'room_type_id', 'room_id', 'allotment_id',
        'group_id', 'posting_account_id', 'reservation_id',
        'tourist_tax_revenue', 'tax_exempt_type',
        'booking_origin_code', 'source_file'
    ]
    # Only drop columns that actually exist in the dataframe to avoid errors
    existing_cols_to_drop = [c for c in cols_to_drop if c in df.columns]
    df = df.drop(columns=existing_cols_to_drop)
    return df

def rename_merge_conflict(df):
    """Maps standardized headers from library output to final target schema."""
    merge_conflict_map = {
      'Market Code': 'pms_segment',
      'Source Code': 'pms_source',
      'Segment Code':'pms_sub_segment'
    }
    # Only rename columns that exist
    existing_merge_conflict_map = {k: v for k, v in merge_conflict_map.items() if k in df.columns}
    df = df.rename(columns=existing_merge_conflict_map)
    return df

def rename_columns(df):
    """Maps standardized headers from library output to final target schema."""
    rename_map = {
      'f_n_b_revenue': 'fb_revenue',
      'rate_code': 'pms_ratecode',
      'sold': 'rms_otb',
      'rev': 'rev_otb',
      'rate_amount': 'rate',
      'original_rate_amount': 'original_rate',
      'reservation_status': 'status',
      'adult_count': 'adults',
      'child_count': 'children',
      'arrival_rate_code': 'arrival_pms_ratecode',
      'first_night_price': 'first_night_rate',
      'last_night_price': 'last_night_rate',
      'departure_room_type': 'departure_roomtype',
      'created_at_hz': 'book_date'
    }
    # Only rename columns that exist
    existing_rename_map = {k: v for k, v in rename_map.items() if k in df.columns}
    df = df.rename(columns=existing_rename_map)
    return df





In [ ]:
import pandas as pd
import re
import os
import shutil
import glob
import numpy as np
from google.colab import drive
import pandas_gbq

# --- EXECUTE SETUP ---
# setup_environment definition is located in the Preflight section (Y66b4AMqXFwD)
setup_environment(
    source_path = "/content/drive/Shareddrives/ClientHubs/Dovetail&Co/pipeline/data_pipeline/process_step03",
    next_path = "/content/drive/Shareddrives/ClientHubs/Dovetail&Co/pipeline/data_pipeline/process_step04"
)

# --- HELPER FUNCTIONS ---
def standardize_headers(df, column_mapping=None):
    """Standardizes headers using external library or fallback logic."""
    if 'revrebel_standardizer' in globals():
        try:
            df = revrebel_standardizer.standardize_columns(df)
            print("  Applied revrebel_standardizer normalization.")
        except Exception as e:
            print(f"  Warning: External standardizer failed ({e}), falling back to basic cleaning.")
            df.columns = [col.lower().strip().replace('/', '_').replace(' ', '_').replace('.', '').replace('-', '_').replace('__', '_') for col in df.columns]
    else:
        df.columns = [col.lower().strip().replace('/', '_').replace(' ', '_').replace('.', '').replace('-', '_').replace('__', '_') for col in df.columns]
    return df

def find_header_row(file_path, keywords=['stay_date', 'confirmation', 'rate_code', 'room', 'hotel_code']):
    """Finds the header row by counting matches for target keywords (sync'd with module logic)."""
    best_row = 0
    max_matches = 0
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            if i > 50: break
            matches = sum(1 for word in keywords if word.lower() in line.lower())
            if matches > max_matches:
                max_matches = matches
                best_row = i
            if max_matches >= 3: return i
    return best_row

# --- MASTER PIPELINE ---
def run_master_pipeline():
    """Orchestrates the data extraction, standardization, and file management."""
    global NEW_DIR, PROCESSED_DIR, EXPORT_DIR, FAILED_DIR, NEXT_DIR

    search_path = os.path.join(NEW_DIR, "*.csv")
    files_to_process = glob.glob(search_path)
    print(f"Found {len(files_to_process)} files in {NEW_DIR}.")

    all_standardized_dfs = []
    last_processed_date = "UNKNOWN"
    last_processed_prop = "UNKNOWN"

    for file_path in files_to_process:
        file_name = os.path.basename(file_path)
        print(f"\nProcessing: {file_name}")

        try:
            date_match = re.search(r"(\d{8})", file_name)
            raw_date = date_match.group(1) if date_match else "UNKNOWN"
            snap_date_str = pd.to_datetime(raw_date, format="%Y%m%d").strftime("%Y-%m-%d") if date_match else "UNKNOWN"

            prop_match = re.search(r"(JFKSNN|NNNH|JFKNOW)", file_name, re.IGNORECASE)
            prop_code = prop_match.group(0).upper() if prop_match else "UNKNOWN"

            if raw_date != "UNKNOWN": last_processed_date = raw_date
            if prop_code != "UNKNOWN": last_processed_prop = prop_code

            header_idx = find_header_row(file_path)
            df = pd.read_csv(file_path, delimiter=',', header=header_idx, on_bad_lines='skip', low_memory=False)

            # STEP 0: Resolve merge conflicts by mapping raw source names first
            if 'rename_merge_conflict' in globals():
                df = rename_merge_conflict(df)
                print("  Applied rename_merge_conflict (pre-standardization).")

            # STEP 1: Normalize column names (Library call)
            df = standardize_headers(df)

            # STEP 2: Drop and Rename (Final cleanup transformations)
            if 'drop_unnecessary_columns' in globals():
                df = drop_unnecessary_columns(df)
                print("  Applied column removal (drop_unnecessary_columns).")

            if 'rename_columns' in globals():
                df = rename_columns(df)
                print("  Applied column renaming (rename_columns). head check: {df.columns[:5]}")

            # STEP 3: Metadata and Placeholder addition
            if 'source_file' in df.columns:
                df = df.drop(columns=['source_file'])

            df['property_code'] = prop_code
            df['snap_date'] = snap_date_str

            new_cols = [
                'channel_code', 'channel', 'channel_sort',
                'segment_code', 'segment', 'segment_sort',
                'source_code', 'source', 'source_sort',
                'subsource_code', 'subsource'
            ]
            for col in new_cols:
                if col not in df.columns:
                    df[col] = ""

            all_standardized_dfs.append(df)
            shutil.move(file_path, os.path.join(PROCESSED_DIR, file_name))
            print(f"  ✅ Success: {prop_code} | Rows: {len(df)} | Moved to Processed")

        except Exception as e:
            print(f"  ❌ Error processing {file_name}: {e}")
            if os.path.exists(file_path):
                shutil.move(file_path, os.path.join(FAILED_DIR, file_name))

    if all_standardized_dfs:
        output_filename = f"{last_processed_date}_{last_processed_prop}_headers_standardized_data.csv"
        export_path = os.path.join(EXPORT_DIR, output_filename)
        next_step_path = os.path.join(NEXT_DIR, output_filename)

        combined_df = pd.concat(all_standardized_dfs, ignore_index=True, sort=False)
        combined_df.to_csv(export_path, index=False)
        combined_df.to_csv(next_step_path, index=False)

        print(f"\nSUCCESS! Exported to: {export_path}")
        display(combined_df.head())

        # --- TRIGGER DOWNSTREAM PIPELINE ---
        print("\n🚀 Triggering Step 04...")
        get_ipython().run_line_magic('run', "'/content/drive/MyDrive/Colab Notebooks/StayInTouch/Step04_StayInTouch_StandardizeData.ipynb'")
    else:
        print("\nNo data found. Ensure files are in the data_upload folder.")

if __name__ == "__main__":
    run_master_pipeline()

--- Initializing Environment ---
Mounted at /content/drive
v Drive mounted successfully.
v Setup complete. Checking for new files in: /content/drive/Shareddrives/ClientHubs/Dovetail&Co/data_pipeline/process_step03/data_upload

Found 1 files in /content/drive/Shareddrives/ClientHubs/Dovetail&Co/data_pipeline/process_step03/data_upload.

Processing: 20260707_JFKNOW_pms_data_merged.csv
  Applied rename_merge_conflict (pre-standardization).
  Applied revrebel_standardizer normalization.
  Applied column removal (drop_unnecessary_columns).
  Applied column renaming (rename_columns). head check: {df.columns[:5]}
  ✅ Success: JFKNOW | Rows: 173292 | Moved to Processed

SUCCESS! Exported to: /content/drive/Shareddrives/ClientHubs/Dovetail&Co/data_pipeline/process_step03/data_export/20260707_JFKNOW_headers_standardized_data.csv


,stay_date,confirmation_number,rate,original_rate,status,roomtype,room_no,block_count,travel_agent,company,...,channel,channel_sort,segment_code,segment,segment_sort,source_code,source,source_sort,subsource_code,subsource
0,2024-12-09,100069.0,0.0,0.0,CHECKEDOUT,WSC,353.0,NaN,Direct Booking,NaN,...,,,,,,,,,,
1,2025-01-01,100000.0,100.0,199.0,CANCELED,ADA,NaN,NaN,NaN,NaN,...,,,,,,,,,,
2,2025-01-02,100000.0,100.0,139.0,CANCELED,ADA,NaN,NaN,NaN,NaN,...,,,,,,,,,,
3,2025-01-07,100197.0,0.0,0.0,CHECKEDOUT,WSC,348.0,NaN,Direct Booking,NaN,...,,,,,,,,,,
4,2025-01-08,100198.0,0.0,0.0,CHECKEDOUT,WSC,318.0,NaN,NaN,NaN,...,,,,,,,,,,



🚀 Triggering Step 04...
Mounted at /content/drive
Drive mounted successfully.
--- Drive Diagnostic ---
v Google Drive is mounted.

Shared Drives found:
 - 'Aparium'
 - 'Backup'
 - 'BCT: Creative Hub'
 - 'ClientHubs'
 - 'Confidential'
 - 'Creative'
 - 'Creative Hub'
 - 'Data'
 - 'External Files'
 - 'Finance'
 - 'Helpfiles'
 - 'Hosted'
 - 'Partners '
 - 'REVREBEL'
 - 'REVREBEL Wiki'
 - 'Stringham Family'
 - 'Templates'
 - 'The Library'
 - 'Toolkits'
 - 'Vault'

v Target path exists: /content/drive/Shareddrives/ClientHubs/Dovetail&Co/data_pipeline/process_step04
Successfully loaded tab: map_rate
Successfully loaded tab: map_crs_channel
Successfully loaded tab: map_crs_subsource
Successfully loaded tab: map_subsource
Successfully loaded tab: map_pms_source
Successfully loaded tab: map_segment
Successfully loaded tab: map_channel
Successfully loaded tab: map_pms_segment
Successfully loaded tab: map_overides
Total tables loaded: 9

--- Table: map_rate ---
Rows: 123


,pms_ratecode,ratecode_name,segment_code,channel_code,channel,source_code,source,subsource_code,subsource
0,ADV,Plan Ahead & Save,UQ,,,,,,
1,BAR,Best Flexible,RE,,,,,,
2,BFCM,Black Friday / Cyber Monday,PR,,,,,,
3,BUDDY,Buddy Buddy Opening Promo,PR,,,,,,
4,CCRP1,CCRP1,UQ,,,,,,



--- Table: map_crs_channel ---
Rows: 20


...


✨ SUCCESS: All criteria met. Data integrity confirmed.

SUCCESS! Modular export complete: 20260707_JFKNOW_standardized_data.csv
Final Dataset row count: 173292


,stay_date,confirmation_number,rate,original_rate,status,roomtype,room_no,block_count,travel_agent,company,...,segment,segment_sort,source_code,source,source_sort,subsource_code,subsource,ratecode_name,segment_group_code,segment_group
0,2024-12-09,100069.0,0.0,0.0,CHECKEDOUT,WSC,353.0,NaN,Direct Booking,NaN,...,Transient Unqualified,16,HD,Hotel Direct,1,HD,Hotel Direct,Plan Ahead & Save,TUQ,Transient Discount
1,2025-01-01,100000.0,100.0,199.0,CANCELED,ADA,NaN,NaN,NaN,NaN,...,Transient Retail,11,HD,Hotel Direct,1,HD,Hotel Direct,Best Flexible,TRE,Transient Retail
2,2025-01-02,100000.0,100.0,139.0,CANCELED,ADA,NaN,NaN,NaN,NaN,...,Transient Retail,11,HD,Hotel Direct,1,HD,Hotel Direct,Best Flexible,TRE,Transient Retail
3,2025-01-07,100197.0,0.0,0.0,CHECKEDOUT,WSC,348.0,NaN,Direct Booking,NaN,...,Transient Unqualified,16,HD,Hotel Direct,1,HD,Hotel Direct,Plan Ahead & Save,TUQ,Transient Discount
4,2025-01-08,100198.0,0.0,0.0,CHECKEDOUT,WSC,318.0,NaN,NaN,NaN,...,Transient Unqualified,16,HD,Hotel Direct,1,HD,Hotel Direct,Plan Ahead & Save,TUQ,Transient Discount


x No files matching '*_standardized*' found in /content/drive/Shareddrives/ClientHubs/Dovetail&Co/data_pipeline/process_step05/data_upload
--- START STEP-BY-STEP DEBUG ---


AttributeError: 'NoneType' object has no attribute 'copy'

AttributeError: 'NoneType' object has no attribute 'copy'

# Veirfy Headers

In [ ]:
import os
import pandas as pd
import glob

# Look for the combined file in the Export directory
target_pattern = "*_headers_standardized_data*"
search_glob = os.path.join(EXPORT_DIR, "**", target_pattern)
found_files = glob.glob(search_glob, recursive=True)

# Filter to valid files and sort by creation time (latest first)
valid_files = [f for f in found_files if os.path.isfile(f)]
valid_files.sort(key=os.path.getctime, reverse=True)

if valid_files:
    file_path = valid_files[0]
    file_name = os.path.basename(file_path)
    print(f"Verifying Latest File: {file_name}")
    print(f"Full Path: {file_path}")

    try:
        # Read the file with low_memory=False to ensure type consistency
        df_check = pd.read_csv(file_path, low_memory=False)

        print("\n--- Data Quality Summary (Target Schema) ---")
        print(f"Total Rows: {len(df_check)}")

        # Target column names after drop/rename logic
        check_cols = [
            'fb_revenue',
            'pms_ratecode',
            'pms_sub_segment',
            'pms_segment',
            'pms_source',
            'rooms_sold',
            'rms_revenue',
            'rate',
            'original_rate',
            'status',
            'arrival_pms_ratecode'
        ]

        for col in check_cols:
            if col in df_check.columns:
                non_null = df_check[col].count()
                pct = (non_null / len(df_check)) * 100
                print(f"Column '{col}': {non_null} non-null values ({pct:.1f}%)")
            else:
                print(f"Column '{col}': NOT FOUND")

        print("\n--- Sample of Processed Data ---")
        # Display a sample of the actual renamed columns
        display_cols = [c for c in check_cols if c in df_check.columns] + ['stay_date', 'property_code']
        display(df_check[display_cols].head(10))

    except Exception as e:
        print(f"❌ Error reading the combined file: {e}")
else:
    print(f"No files matching pattern found in: {EXPORT_DIR}. Please run the master pipeline cell (zDkwBwlw-CyM) first.")

In [ ]:
import pandas as pd
import os

# Get the latest standardized file path from the previous verification steps
if 'file_path' in globals() and os.path.exists(file_path):
    # Read only the header row to be efficient
    df_headers = pd.read_csv(file_path, nrows=0)

    # Create a DataFrame of column names for clear display
    headers_display_df = pd.DataFrame(df_headers.columns, columns=['Standardized Header'])

    print(f"Displaying headers for: {os.path.basename(file_path)}")
    display(headers_display_df)
else:
    print("Error: No processed file found to extract headers from. Please ensure the pipeline has run successfully.")

In [ ]:
# FINAL HEADER INSPECTION
import pandas as pd
import os

if 'file_path' in globals() and os.path.exists(file_path):
    df_final = pd.read_csv(file_path, nrows=0)
    cols = sorted(df_final.columns.tolist())

    print(f"Final Exported Column Count: {len(cols)}")
    print("--- List of Final Standardized Headers ---")
    for i, col in enumerate(cols, 1):
        print(f"{i:02d}. {col}")
else:
    print("Process complete. Run the verification cell above to load the file path.")

# Validation / Diagnostics
This section verifies the integrity of the merged data against the success criteria defined for January 2026.

In [ ]:
import pandas as pd
import os
import glob

# --- Validation Module ---
def run_success_criteria_test(df):
    print("--- Running Success Criteria Test ---")

    # Filtering strictly on the 'Date' column as requested
    date_col = 'stay_date'

    if date_col not in df.columns:
         print(f"❌ Error: Required column '{date_col}' not found in the dataset.")
         return

    print(f"Filtering based on column: {date_col}")
    df[date_col] = pd.to_datetime(df[date_col])

    # 1. Define Filters (Criteria uhfOwqak033W)
    start_date = '2026-01-01'
    end_date = '2026-01-31'
    status_filter = 'CHECKEDOUT'

    # 2. Apply Filters
    mask = (
        (df[date_col] >= start_date) &
        (df[date_col] <= end_date) &
        (df['status'] == status_filter)
    )
    test_df = df.loc[mask].copy()

    # 3. Calculate Sums
    total_sold = pd.to_numeric(test_df['rms_otb'], errors='coerce').sum()
    total_revenue = pd.to_numeric(test_df['rev_otb'], errors='coerce').sum()

    # 4. Success Criteria Targets
    target_sold = 2235
    target_revenue = 175872

    # 5. Display Results with Tolerance for Decimals
    sold_pass = int(round(total_sold)) == target_sold
    rev_pass = abs(total_revenue - target_revenue) < 1.0

    results_data = {
        "Metric": ["Sum Col 'rms_otb'", "Sum Col 'rev_otb'"],
        "Actual": [round(total_sold, 2), round(total_revenue, 2)],
        "Target": [target_sold, target_revenue],
        "Status": [
            "✅ PASS" if sold_pass else "❌ FAIL",
            "✅ PASS" if rev_pass else "❌ FAIL"
        ]
    }

    results_df = pd.DataFrame(results_data)
    display(results_df)

    if sold_pass and rev_pass:
        print("\n✨ SUCCESS: All criteria met. Data integrity confirmed.")
    else:
        print("\n⚠️ WARNING: Criteria mismatch. Check if the dates or filters need adjustment.")

# Execution Logic
if 'combined_df' in locals():
    run_success_criteria_test(combined_df)
elif 'EXPORT_DIR' in globals():
    files = glob.glob(os.path.join(EXPORT_DIR, "*_headers_standardized_data.csv"))
    if files:
        latest_export = max(files, key=os.path.getmtime)
        print(f"Loading latest export for validation: {os.path.basename(latest_export)}")
        loaded_df = pd.read_csv(latest_export, low_memory=False)
        run_success_criteria_test(loaded_df)
    else:
        print("❌ Error: No exported files found in EXPORT_DIR.")
else:
    print("❌ Error: combined_df not found and EXPORT_DIR not defined.")